# ETF notebook (event-driven engine)

This notebook is meant for quick manual backtests and (later) manual trade wiring.

- Uses the **event-driven** core (`Strategy.on_bar`) for realistic execution parity with IBKR later.
- Uses **Yahoo Finance via `pandas_datareader`** with a **parquet cache partitioned by date** under `data/prices/yahoo/`.
- Prices are streamed in **date batches** to avoid large in-memory datasets.

In [1]:
import sys
from pathlib import Path

# Make `algotrade/src` importable when running from notebooks/
repo_root = Path.cwd().parents[0]
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root:', repo_root)
print('src_path:', src_path)

repo_root: /home/dnguyen3/code/algotrade
src_path: /home/dnguyen3/code/algotrade/src


In [4]:
# If needed, install these in your env:
# !pip install pandas numpy pyyaml pandas_datareader pyarrow

from backtest.backtester import run_backtest
from strategies.momentum import SMAMomentum

symbol = 'SPY'
strategy = SMAMomentum(
    strategy_id='nb_sma_momentum',
    fast_window=50,
    slow_window=200,
    symbol=symbol,
    max_position_pct=0.95,
)

portfolio_cfg = {'base_currency': 'USD', 'virtual_cash': 30000}
risk_cfg = {'max_single_position_pct': 1.0, 'max_drawdown_pct': 0.25}
storage_path = str(repo_root / 'trades.sqlite3')

price_source = {
    'type': 'yahoo',
    'start': '2024-12-31',
    'end': '2015-01-01',
    'batch_days': 90,
    'prefer_adj_close': True,
}

results = run_backtest(strategy, price_source, portfolio_cfg, risk_cfg, storage_path)
results

{'final_equity': 30000, 'cash': 30000, 'positions': {}}

In [5]:
from market_data.prices import YahooPriceStore

store = YahooPriceStore.default()
cache_root = store.data_dir / 'prices' / 'yahoo' / symbol.upper()
print('Cache root:', cache_root)

# Show some cached partitions (date=YYYY-MM-DD folders)
if cache_root.exists():
    parts = sorted([p.name for p in cache_root.iterdir() if p.is_dir() and p.name.startswith('date=')])
    print('Partitions:', parts[:5], '... total:', len(parts))
else:
    print('Cache folder does not exist yet (run the backtest cell above).')

Cache root: /home/dnguyen3/code/algotrade/data/prices/yahoo/SPY
Cache folder does not exist yet (run the backtest cell above).
